# Wan2.1 I2V 14B GGUF — API Server
**Using ComfyUI + Q4 GGUF on free T4 | Portrait 9:16 for Reels/Shorts**

Run all cells → wait for **API URL** → paste in Phase 6 of your project.

In [ ]:
# @title 📦 1. Install Dependencies
!pip install torch==2.6.0 torchvision==0.21.0 -q
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install -q av fastapi uvicorn nest-asyncio
!apt -y install -qq aria2 ffmpeg

import os, torch, gc, uuid, time, threading, asyncio, subprocess, json, random
from pathlib import Path
from PIL import Image
import numpy as np
import imageio

In [ ]:
# @title 🤖 2. Setup ComfyUI + GGUF
%cd /content

# Clone ComfyUI
!git clone https://github.com/Isi-dev/ComfyUI

# Clone GGUF custom node
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI

# Download GGUF models (Q4_0 = ~8GB, fast on T4)
print("Downloading Q4 GGUF model...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-Q4_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-i2v-14b-480p-Q4_0.gguf

# Download support files
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors -d /content/ComfyUI/models/clip_vision -o clip_vision_h.safetensors

print("✅ All models downloaded!")

In [ ]:
# @title 🚀 3. Start API Server
import sys
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management
from nodes import (
    CheckpointLoaderSimple, CLIPLoader, CLIPTextEncode,
    VAEDecode, VAELoader, KSampler, LoadImage,
    CLIPVisionLoader, CLIPVisionEncode
)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_video import SaveWEBM
from comfy_extras.nodes_wan import WanImageToVideo

# Initialize nodes once
unet_loader = UnetLoaderGGUF()
model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_pos = CLIPTextEncode()
clip_encode_neg = CLIPTextEncode()
vae_loader = VAELoader()
clip_vision_loader = CLIPVisionLoader()
clip_vision_encode = CLIPVisionEncode()
load_image = LoadImage()
wan_image_to_video = WanImageToVideo()
ksampler = KSampler()
vae_decode = VAEDecode()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def generate_video_internal(image_path, prompt, seed, width, height, frames=49, steps=20):
    with torch.inference_mode():
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_pos.encode(clip, prompt)[0]
        negative = clip_encode_neg.encode(clip, "色调艳丽，过曝，静态，细节模糊不清，字幕，最差质量，低质量，丑陋的，畸形的")[0]
        del clip
        torch.cuda.empty_cache()

        loaded_image = load_image.load_image(image_path)[0]
        clip_vision = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
        clip_vision_output = clip_vision_encode.encode(clip_vision, loaded_image, "none")[0]
        del clip_vision
        torch.cuda.empty_cache()

        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        positive_out, negative_out, latent = wan_image_to_video.encode(
            positive, negative, vae, width, height, frames, 1, loaded_image, clip_vision_output
        )

        model = unet_loader.load_unet("wan2.1-i2v-14b-480p-Q4_0.gguf")[0]
        model = model_sampling.patch(model, 8)[0]

        sampled = ksampler.sample(
            model=model, seed=seed, steps=steps, cfg=1.0,
            sampler_name="uni_pc", scheduler="simple",
            positive=positive_out, negative=negative_out, latent_image=latent
        )[0]
        del model
        torch.cuda.empty_cache()

        decoded = vae_decode.decode(vae, sampled)[0]
        del vae
        torch.cuda.empty_cache()

        frames_list = [(img.cpu().numpy() * 255).astype(np.uint8) for img in decoded]
        return frames_list

import nest_asyncio
nest_asyncio.apply()

!pkill -f cloudflared 2>/dev/null; sleep 1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()
tasks = {}
API_URL = None

def start_tunnel():
    global API_URL
    proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080"],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
    )
    for line in iter(proc.stderr.readline, b''):
        txt = line.decode().strip()
        if "trycloudflare.com" in txt:
            API_URL = "https://" + txt.split("https://")[-1].strip()
            print(f"\n✅  API URL: {API_URL}")
            print("⚠️  Copy this URL and paste it in Phase 6 of your project!\n")
            break

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/generate")
async def generate(image: UploadFile = File(...), prompt: str = Form(...)):
    task_id = uuid.uuid4().hex[:8]
    img = Image.open(image.file).convert("RGB")
    img = img.resize((480, 848))  # Portrait ratio for Reels
    input_path = f"/content/input_{task_id}.png"
    img.save(input_path)
    tasks[task_id] = {"state": "queued"}
    threading.Thread(target=_run, args=(task_id, prompt, input_path), daemon=True).start()
    return {"task_id": task_id}

def _run(task_id, prompt, input_path):
    tasks[task_id]["state"] = "running"
    try:
        frames_list = generate_video_internal(
            image_path=input_path,
            prompt=prompt,
            seed=random.randint(0, 2**32 - 1),
            width=480, height=848,  # Portrait 9:16
            frames=49, steps=20
        )
        output_path = f"/content/output_{task_id}.mp4"
        with imageio.get_writer(output_path, fps=16) as writer:
            for frame in frames_list:
                writer.append_data(frame)
        tasks[task_id]["state"] = "completed"
        tasks[task_id]["video_url"] = f"{API_URL}/video/{task_id}"
    except Exception as e:
        tasks[task_id]["state"] = "failed"
        tasks[task_id]["error"] = str(e)

@app.get("/status/{task_id}")
def status(task_id: str):
    return tasks.get(task_id, {"state": "unknown"})

@app.get("/video/{task_id}")
def video(task_id: str):
    path = f"/content/output_{task_id}.mp4"
    if os.path.exists(path):
        return FileResponse(path, media_type="video/mp4")
    return {"error": "not found"}

# Start tunnel + server
threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8080, log_level="error")
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.create_task(server.serve())
print("✅ Server running!")

while API_URL is None:
    time.sleep(1)

---
## ✅ Done!
1. Copy the **API URL** above
2. Paste in Phase 6 of main.py
3. Each video: ~26 min on T4 (Q4) — **480x848 portrait** for Reels

⚠️ Keep this Colab tab open